# 04 · CNNs on Digits

Write 2-D convolution and pooling from scratch, use them to find edges in a photo, derive the output-size formula,
then train a small CNN on 8×8 handwritten digits and compare it with an MLP on parameters, accuracy and robustness to
a one-pixel shift. It ends where modern vision models start: the patch embedding of a Vision Transformer, which turns
out to be a convolution.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [02 · training loop](02_training_loop_optimizers_lr_schedules.ipynb)

## Why this matters in interviews

- "What does a convolution do, and why use one instead of a fully connected layer?" Weight sharing and locality,
  with a parameter count to prove it.
- The output-size formula `⌊(n + 2p − d(k−1) − 1)/s⌋ + 1` and the conv parameter count come up in live coding and
  in "debug this shape error" questions.
- Multimodal LLM questions start here: a vision encoder cuts an image into patches with a strided convolution, and
  every patch becomes a token, which is why images are expensive in tokens.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `fo28` | What is multimodality, and what changes when you add images? (patch embeddings, image token counts) |
| `fo06` | What are parameters, and does a bigger parameter count mean a better model? (a CNN beats an MLP with fewer) |
| `fo11` | What is the difference between AI, machine learning, deep learning and generative AI? (learned features vs hand-made ones) |

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import convolve2d, correlate2d
from sklearn.datasets import load_digits, load_sample_image
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

np.random.seed(0); torch.manual_seed(0)
torch.set_num_threads(1)                         # tiny models: single-threaded is fastest
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=COLORS), "axes.grid": True, "grid.color": "#e1e0d9",
                     "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1.8})

photo = load_sample_image("china.jpg").mean(axis=2)[::4, ::4] / 255.0   # bundled with scikit-learn, no download
print(f"grayscale photo {photo.shape}, values in [{photo.min():.2f}, {photo.max():.2f}]")

## 1. Convolution from scratch

**In plain English:** slide a small grid of weights (the *kernel*, here 3×3) over the image. At every position,
multiply the kernel by the patch of pixels under it and add everything up; that sum is one output pixel. A kernel
is a tiny pattern detector: the output is large wherever the image locally looks like the kernel.

Deep-learning "convolution" (`nn.Conv2d`) is technically *cross-correlation*: the kernel is not flipped. The
mathematical convolution (`scipy.signal.convolve2d`) flips it. For learned kernels the difference is irrelevant, but
the asserts below pin down which is which.

In [ ]:
def conv2d(img, kernel, stride=1, padding=0):
    """Naive 2-D cross-correlation, exactly what nn.Conv2d computes for one channel."""
    if padding:
        img = np.pad(img, padding)                   # zero padding on every side
    kh, kw = kernel.shape
    out_h = (img.shape[0] - kh) // stride + 1
    out_w = (img.shape[1] - kw) // stride + 1
    out = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            patch = img[i * stride:i * stride + kh, j * stride:j * stride + kw]
            out[i, j] = (patch * kernel).sum()       # multiply element-wise, then sum
    return out

sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)   # responds to left-right brightness change
sobel_y = sobel_x.T                                                      # responds to top-bottom change

t0 = time.time()
edges_v = conv2d(photo, sobel_x)
edges_h = conv2d(photo, sobel_y)
print(f"input {photo.shape} -> output {edges_v.shape} in {time.time() - t0:.2f}s (pure Python loops)")

torch_out = F.conv2d(torch.tensor(photo)[None, None], torch.tensor(sobel_x)[None, None])[0, 0].numpy()
assert np.allclose(edges_v, torch_out)                                    # same as PyTorch's conv
assert np.allclose(edges_v, correlate2d(photo, sobel_x, mode="valid"))    # = cross-correlation
assert np.allclose(edges_v, convolve2d(photo, sobel_x[::-1, ::-1], mode="valid"))   # = true convolution, flipped kernel
print("matches F.conv2d and scipy correlate2d; true convolution needs the kernel flipped")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 4.5))
for row, (kernel, out, name) in enumerate([(sobel_x, edges_v, "vertical edges"), (sobel_y, edges_h, "horizontal edges")]):
    axes[row, 0].imshow(photo, cmap="gray"); axes[row, 0].set_title("input")
    axes[row, 1].imshow(kernel, cmap="coolwarm", vmin=-2, vmax=2)
    for (i, j), v in np.ndenumerate(kernel):
        axes[row, 1].text(j, i, f"{v:+.0f}", ha="center", va="center", fontsize=11)
    axes[row, 1].set_title(f"kernel: Sobel ({name})")
    lim = np.abs(out).max() / 2
    axes[row, 2].imshow(out, cmap="coolwarm", vmin=-lim, vmax=lim)
    axes[row, 2].set_title(f"output: {name} light up")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout(); plt.show()

The vertical-edge kernel (−1 on the left, +1 on the right) lights up where brightness changes from left to right:
the sides of the tower and the tree trunks. Its transpose finds horizontal edges: the stacked roof lines. Red and blue are the
two directions of change. A CNN *learns* kernels like these from data instead of having them hand-designed, which
is the step from classic computer vision (hand-made features) to deep learning (learned features).

## 2. Stride, padding, dilation: the output-size formula

$$\text{out} = \left\lfloor \frac{n + 2p - d\,(k-1) - 1}{s} \right\rfloor + 1$$

for input size `n`, kernel `k`, padding `p`, stride `s`, dilation `d` (with `d = 1` this is the familiar
`⌊(n + 2p − k)/s⌋ + 1`). Check it against PyTorch for every combination in a grid:

In [ ]:
def conv_out(n, k, s=1, p=0, d=1):
    return (n + 2 * p - d * (k - 1) - 1) // s + 1

checked = 0
for n in [7, 8, 28, 32]:
    for k in [1, 3, 5]:
        for s in [1, 2, 3]:
            for p in [0, 1, 2]:
                for d in [1, 2]:
                    if conv_out(n, k, s, p, d) < 1:
                        continue
                    conv = nn.Conv2d(1, 1, kernel_size=k, stride=s, padding=p, dilation=d)
                    assert conv(torch.zeros(1, 1, n, n)).shape[-1] == conv_out(n, k, s, p, d), (n, k, s, p, d)
                    checked += 1
assert conv2d(np.zeros((8, 8)), np.ones((3, 3)), stride=2, padding=1).shape == (conv_out(8, 3, 2, 1),) * 2
print(f"formula matches nn.Conv2d on {checked} (n, k, s, p, d) combinations")
print("'same' padding for k=3, s=1:", conv_out(28, 3, 1, 1), "| stride 2 halves:", conv_out(28, 3, 2, 1),
      "| no padding shrinks by k-1:", conv_out(28, 3))

conv = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3)
by_formula = 64 * (3 * 3 * 3 + 1)                # C_out * (C_in * k * k + 1 bias)
print(f"Conv2d(3 -> 64, 3x3) parameters: {sum(p.numel() for p in conv.parameters()):,} = 64 x (3*3*3 + 1)")
assert sum(p.numel() for p in conv.parameters()) == by_formula

Notice what the parameter count does **not** depend on: the image size. The same 1,792 weights process a 32×32 or a
4000×3000 image, because one small kernel is reused at every position (**weight sharing**) and only looks at a small
neighbourhood (**locality**). A fully connected layer from a 224×224×3 image to just 64 units would need 9.6 million
weights. Stacking helps too: two 3×3 layers see a 5×5 region (18 weights per channel pair instead of 25).

## 3. Pooling

Pooling shrinks a feature map by summarising each window: max pooling keeps the strongest response ("the edge is
somewhere in here"), average pooling keeps the mean. It has no parameters, and makes the output less sensitive to
exactly where a feature sits.

In [ ]:
fmap = np.array([[1, 3, 2, 0],
                 [4, 6, 1, 1],
                 [0, 2, 9, 5],
                 [1, 1, 3, 7]], dtype=float)

def pool2d(x, size=2, op=np.max):
    h, w = x.shape[0] // size, x.shape[1] // size
    return np.array([[op(x[i * size:(i + 1) * size, j * size:(j + 1) * size]) for j in range(w)] for i in range(h)])

max_p, avg_p = pool2d(fmap), pool2d(fmap, op=np.mean)
print("max pool 2x2:\n", max_p, "\navg pool 2x2:\n", avg_p)
t = torch.tensor(fmap)[None, None]
assert np.allclose(max_p, F.max_pool2d(t, 2)[0, 0].numpy()) and np.allclose(avg_p, F.avg_pool2d(t, 2)[0, 0].numpy())

## 4. A small CNN vs an MLP on 8×8 digits

`load_digits` has 1,797 handwritten digits of 8×8 pixels. The CNN keeps the image as a 2-D grid; the MLP flattens it
into 64 unrelated numbers.

```text
CNN: [1,8,8] -conv3x3-> [8,8,8] -ReLU-maxpool-> [8,4,4] -conv3x3-> [16,4,4] -ReLU-maxpool-> [16,2,2] -flatten-> 64 -linear-> 10
MLP: [64] -linear-> 64 -ReLU-> -linear-> 10
```

In [ ]:
digits = load_digits()
images = digits.images / 16.0                                 # [1797, 8, 8], pixel values scaled to [0, 1]
X_tr, X_te, y_tr, y_te = train_test_split(images, digits.target, test_size=0.25, random_state=0,
                                          stratify=digits.target)
X_tr, X_te = torch.tensor(X_tr, dtype=torch.float32)[:, None], torch.tensor(X_te, dtype=torch.float32)[:, None]
y_tr, y_te = torch.tensor(y_tr), torch.tensor(y_te)
print(f"train {tuple(X_tr.shape)}   test {tuple(X_te.shape)}   ([N, channels, height, width])")

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)     # 8 learned 3x3 filters
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.head = nn.Linear(16 * 2 * 2, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)                # [N, 8, 4, 4]
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)                # [N, 16, 2, 2]
        return self.head(x.flatten(1))

def make_mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))

def n_params(model):
    return sum(p.numel() for p in model.parameters())

torch.manual_seed(0); cnn = SmallCNN()
torch.manual_seed(0); mlp = make_mlp()
print(f"CNN parameters: {n_params(cnn):,}   MLP parameters: {n_params(mlp):,}")

In [ ]:
def train_model(model, epochs=12, lr=1e-2, batch_size=128):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    gen, losses = torch.Generator().manual_seed(0), []
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(X_tr), generator=gen)
        for i in range(0, len(X_tr), batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            loss = F.cross_entropy(model(X_tr[idx]), y_tr[idx])
            loss.backward()
            opt.step()
            losses.append(loss.item())
    return losses

def accuracy(model, X, Y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(1) == Y).float().mean().item()

curves, secs = {}, {}
for name, model in [("CNN", cnn), ("MLP", mlp)]:
    t0 = time.time()
    curves[name] = train_model(model)
    secs[name] = time.time() - t0

fig, ax = plt.subplots(figsize=(10, 3.8))
for name, c in curves.items():
    ax.plot(np.convolve(c, np.ones(10) / 10, mode="valid"), label=name)
ax.set(title="Training loss (10-step moving average)", xlabel="step", ylabel="cross-entropy", yscale="log")
ax.legend(); plt.show()

Both learn fast. Now the comparison, including a robustness test: shift every test image **one pixel to the
right**. A person would not notice; a model that has learned "stroke shapes" should not either.

In [ ]:
def shift_right(X, pixels=1):
    out = torch.zeros_like(X)
    out[..., pixels:] = X[..., :-pixels]
    return out

X_te_shift = shift_right(X_te)
rows = {name: (n_params(m), accuracy(m, X_te, y_te), accuracy(m, X_te_shift, y_te), secs[name])
        for name, m in [("CNN", cnn), ("MLP", mlp)]}
print(f"{'model':6s} {'params':>7s} {'test acc':>9s} {'shifted 1px':>12s} {'train time':>11s}")
for name, (n, acc, acc_s, s) in rows.items():
    print(f"{name:6s} {n:7,d} {acc:9.3f} {acc_s:12.3f} {s:10.1f}s")
assert rows["CNN"][0] < rows["MLP"][0] / 2                  # fewer than half the parameters ...
assert rows["CNN"][1] >= rows["MLP"][1] - 0.01              # ... for (at least) the same accuracy
assert rows["CNN"][2] > rows["MLP"][2] + 0.05               # and far more robust to a 1-pixel shift

On 8×8 digits both models are accurate (the task is easy; do not expect a big gap on clean data). The CNN gets
there with well under half the parameters, and it degrades much less when the image moves by one pixel: the same
kernel is applied at every position, so a shifted stroke produces a shifted feature map, and max pooling absorbs
small shifts. The MLP learned "pixel 27 is bright" style rules; move the digit and those pixels change. That built-in
assumption (*inductive bias*: locality and translation equivariance) is what CNNs buy you, and it matters most when
data is limited.

## 5. What the CNN learned

The 8 first-layer kernels, and what each produces for one test digit (the *feature maps*):

In [ ]:
kernels = cnn.conv1.weight.detach()[:, 0]                        # [8, 3, 3]
digit = X_te[:1]
with torch.no_grad():
    maps = F.relu(cnn.conv1(digit))[0]                           # [8, 8, 8]

fig, axes = plt.subplots(2, 9, figsize=(11, 3))
axes[0, 0].imshow(digit[0, 0], cmap="gray_r"); axes[0, 0].set_title(f"input: {y_te[0].item()}", fontsize=9)
axes[1, 0].axis("off")
lim = kernels.abs().max().item()
for c in range(8):
    axes[0, c + 1].imshow(kernels[c], cmap="coolwarm", vmin=-lim, vmax=lim)
    axes[0, c + 1].set_title(f"kernel {c}", fontsize=9)
    axes[1, c + 1].imshow(maps[c], cmap="gray_r")
    axes[1, c + 1].set_title(f"map {c}", fontsize=9)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.tight_layout(); plt.show()

Top row: learned 3×3 kernels (red positive, blue negative weights); many look like oriented edge or stroke
detectors, the same family as the Sobel kernels above, found by gradient descent. Bottom row: where each fires on
this digit. Deeper layers combine these into strokes, loops and corners.

Where does the CNN still go wrong? A confusion matrix counts (true digit, predicted digit) pairs:

In [ ]:
cnn.eval()
with torch.no_grad():
    pred = cnn(X_te).argmax(1).numpy()
cm = confusion_matrix(y_te.numpy(), pred)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    if v:
        ax.text(j, i, v, ha="center", va="center", fontsize=8, color="white" if v > cm.max() / 2 else "#0b0b0b")
ax.set(title="CNN confusion matrix (test set)", xlabel="predicted digit", ylabel="true digit",
       xticks=range(10), yticks=range(10))
ax.grid(False); plt.show()
off_diag = cm - np.diag(np.diag(cm))
i, j = np.unravel_index(off_diag.argmax(), cm.shape)
print(f"{cm.trace()} of {cm.sum()} correct; most common mistake: true {i} predicted as {j} ({off_diag[i, j]} times)")
assert cm.sum() == len(y_te) and math.isclose(cm.trace() / cm.sum(), rows["CNN"][1], abs_tol=1e-6)

## 6. From CNNs to Vision Transformers: the patch embedding is a convolution

A Vision Transformer (ViT) cuts the image into non-overlapping patches (16×16 pixels for ViT-B/16), flattens each
patch and projects it with one linear layer to a `d_model`-sized vector: one token per patch. A `Conv2d` with
`kernel_size = stride = patch size` does exactly that, which is how every implementation writes it:

In [ ]:
torch.manual_seed(0)
patch, d_model = 16, 64
img = torch.randn(1, 3, 224, 224)
patch_embed = nn.Conv2d(3, d_model, kernel_size=patch, stride=patch)
tokens = patch_embed(img).flatten(2).transpose(1, 2)              # [1, 64, 14, 14] -> [1, 196, 64]
print("image", tuple(img.shape), "-> tokens", tuple(tokens.shape))

patches = F.unfold(img, kernel_size=patch, stride=patch).transpose(1, 2)            # [1, 196, 3*16*16]
as_linear = patches @ patch_embed.weight.reshape(d_model, -1).T + patch_embed.bias   # flatten + Linear
assert torch.allclose(tokens, as_linear, atol=1e-4)
print("Conv2d(kernel=stride=16) == flatten each 16x16 patch + one Linear layer:", True)

for size, p in [(224, 16), (224, 14), (336, 14)]:
    print(f"{size}x{size} image, {p}x{p} patches -> {(size // p) ** 2} tokens")
assert (336 // 14) ** 2 == 576          # the CLIP ViT-L/14-336 encoder used by LLaVA-1.5: 576 image tokens

After the patch embedding, a ViT is just a transformer over those tokens (plus position embeddings), with no further
convolutions. The trade-off:

| | CNN | Vision Transformer |
|---|---|---|
| built-in assumption | locality + translation equivariance (strong inductive bias) | almost none: attention is global from layer 1 |
| data needed | works well on small and medium datasets | needs large pretraining data (or distillation) to beat CNNs |
| compute vs resolution | linear in pixels | quadratic in the number of patches (attention) |
| typical uses | edge devices, detection backbones (ResNet, EfficientNet, ConvNeXt) | CLIP, SigLIP, DINOv2, the vision encoders in multimodal LLMs |

For multimodal LLMs the consequence is cost: the image becomes hundreds of tokens (576 for a 336-pixel CLIP
ViT-L/14 image), more with higher resolution or tiling, all of which the language model must attend over.

## Try it yourself

**1.** Design a 3×3 kernel that blurs (averages the neighbourhood) and one that sharpens (centre 5, the four
neighbours −1). Apply both to the photo with `conv2d(..., padding=1)` and check the output keeps the input's shape.

In [ ]:
# Solution — try it yourself first, then run this
blur = np.ones((3, 3)) / 9
sharpen = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=float)
small = photo[40:80, 60:110]                                     # a crop keeps the pure-Python loop fast
blurred, sharpened = conv2d(small, blur, padding=1), conv2d(small, sharpen, padding=1)
assert blurred.shape == sharpened.shape == small.shape            # 'same' padding: p = (k - 1) / 2
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, im, t in zip(axes, [small, blurred, sharpened], ["crop", "blur (box 3x3)", "sharpen"]):
    ax.imshow(im, cmap="gray", vmin=0, vmax=1); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()
print(f"pixel-to-pixel variation: original {np.abs(np.diff(small)).mean():.4f}, "
      f"blurred {np.abs(np.diff(blurred)).mean():.4f}, sharpened {np.abs(np.diff(sharpened)).mean():.4f}")
assert np.abs(np.diff(blurred)).mean() < np.abs(np.diff(small)).mean() < np.abs(np.diff(sharpened)).mean()

**2.** Compute, without running a model, the output shape and parameter count of `Conv2d(64, 128, kernel_size=3,
stride=2, padding=1)` applied to a `[8, 64, 56, 56]` batch. Then verify.

In [ ]:
# Solution — try it yourself first, then run this
out_hw = conv_out(56, k=3, s=2, p=1)
params = 128 * (64 * 3 * 3 + 1)
layer = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
out = layer(torch.zeros(8, 64, 56, 56))
print(f"predicted [8, 128, {out_hw}, {out_hw}] with {params:,} params; actual {list(out.shape)} with {n_params(layer):,}")
assert list(out.shape) == [8, 128, out_hw, out_hw] and n_params(layer) == params

**3.** Data augmentation fixes the MLP's weakness: train a fresh MLP on the training images *plus* copies shifted one
pixel left and right, and test it on the shifted test set again.

In [ ]:
# Solution — try it yourself first, then run this
def shift_left(X, pixels=1):
    out = torch.zeros_like(X)
    out[..., :-pixels] = X[..., pixels:]
    return out

X_plain, y_plain = X_tr, y_tr
X_tr = torch.cat([X_plain, shift_right(X_plain), shift_left(X_plain)])   # 3x the data, same labels
y_tr = torch.cat([y_plain, y_plain, y_plain])
torch.manual_seed(0); mlp_aug = make_mlp()
train_model(mlp_aug, epochs=10)
X_tr, y_tr = X_plain, y_plain                                             # restore the original training set
acc_aug = accuracy(mlp_aug, X_te_shift, y_te)
print(f"MLP on shifted test images: {rows['MLP'][2]:.3f} without augmentation -> {acc_aug:.3f} with it")
assert acc_aug > rows["MLP"][2] + 0.05

## Say it in an interview

- **Convolution:** slide a small learned kernel over the input, dot product at each position. Two properties
  explain everything: *locality* (each output looks at a small neighbourhood) and *weight sharing* (the same kernel
  everywhere), so parameters are `C_out × (C_in × k × k + 1)`, independent of image size.
- **Shapes:** `out = ⌊(n + 2p − d(k−1) − 1)/s⌋ + 1`; 'same' padding for odd `k` is `p = (k−1)/2`; stride 2 halves
  the resolution. Pooling downsamples with no parameters.
- **Why CNNs beat MLPs on images:** far fewer parameters for the same accuracy and robustness to small shifts
  (shown above: one pixel of shift hurts the MLP much more). The inductive bias substitutes for data.
- **nn.Conv2d is cross-correlation** (no kernel flip); irrelevant for learning, a classic trivia follow-up.
- **CNN → ViT:** a ViT's first layer is a `Conv2d` with kernel = stride = patch size, i.e. flatten each patch and
  apply one linear layer; then a plain transformer over patch tokens. ViTs need more data but scale better and are
  the vision encoders in multimodal LLMs.
- **Image token cost:** (side / patch)² tokens per image, e.g. 224/16 → 196, 336/14 → 576. Resolution is a cost knob.
- **Traps:** forgetting the channel dimension (`[N, C, H, W]`, a grayscale image still needs `C = 1`); counting
  conv parameters with the image size; claiming max pooling makes a CNN fully translation *invariant* (it only
  tolerates small shifts; augmentation does the rest).

## Next

- [05 · Embeddings, RNNs and the road to attention](05_embeddings_rnn_to_attention.ipynb): from grids to sequences.
- [03 · Regularization and normalization](03_regularization_dropout_batchnorm.ipynb): BatchNorm, the norm CNNs use.
- [Attention from scratch](../07_genai_foundations/03_attention_from_scratch.ipynb): what a ViT does with its patch
  tokens · [Embeddings and similarity](../07_genai_foundations/02_embeddings_and_similarity.ipynb)
- Theory: [dl_fundamentals course](../../dl_fundamentals/index.html) (chapter 9, convolution) ·
  [interview bank](../../ai_interview_prep/index.html)